# VTuber Builder — 순서대로 실행

**① 환경 준비(다운로드·설치·모델 검증)** → **② Gradio UI 실행** → UI에서 **③ 캐릭터 생성 / ④ 액세서리 적용**. Colab의 GPU RAM이 0인 동안에는 설치·다운로드·CPU 검증 중일 수 있습니다. 모델 추론 시작 및 GPU 사용률은 UI에서 확인합니다.

## ① GitHub · 패키지 · 모델 준비

아래 셀에서 실시간 설치 로그를 전부 확인할 수 있습니다. 성공 메시지 `✅ ① 준비 완료`가 나오기 전에는 다음 셀을 실행하지 마세요.

In [ ]:
import pathlib, shutil, subprocess, sys, runpy

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")

def run(cmd, timeout):
    print(f"[Colab 준비] $ {' '.join(cmd)}", flush=True)
    # No capture_output: git, pip, model downloads and errors remain visible.
    subprocess.run(cmd, check=True, text=True, timeout=timeout)

print("①-1 / GitHub main 다운로드 및 동기화", flush=True)
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)

run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)
print("①-2 / Gradio UI 패키지 설치", flush=True)
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "gradio==6.3.0"], 600)

print("①-3 / 런타임·모델 다운로드·무결성·라이선스 검증 (GPU 추론 이전)", flush=True)
# Install in a CHILD Python process. Pip changes installed packages during
# preparation; the notebook kernel must not retain stale Gradio/NumPy imports.
setup_python = (
    "import pathlib, runpy; "
    "REPO_DIR = pathlib.Path('/content/Virtual-pipeline'); "
    "app = runpy.run_path(str(REPO_DIR / 'tools' / 'colab_app.py'), "
    "run_name='vtuber_prepare'); "
    "head, logs = app['ensure_runtime'](); "
    "print('\\n'.join(logs), flush=True); "
    "print('✅ ① 준비 완료. main=' + head[:12], flush=True)"
)
run([sys.executable, "-u", "-c", setup_python], 9000)
print("✅ 이제 아래 ② 셀을 실행하세요.", flush=True)
print("전체 설치 로그: /content/vtuber_builder/logs/runtime_setup.log", flush=True)


## ② 생성 웹 UI 실행

①이 완료된 후 실행하세요. 아래 UI에서 작업 모드를 고르고 이미지를 올리면 단계별 상태, GPU 메모리와 전체 로그 다운로드가 보입니다.

In [ ]:
import pathlib, runpy

repo = pathlib.Path("/content/Virtual-pipeline")
print("② 웹 UI 실행 · Avatar/Accessory 입력 및 실시간 제작 로그", flush=True)
print("생성 버튼은 설치 작업을 수행하지 않습니다.", flush=True)
runpy.run_path(str(repo / "tools" / "colab_app.py"), run_name="__main__")
